# 02 — Quick Ordinal Baseline (Smoke Run)

Tiny-subset run to verify the Phase 4 pipeline before submitting the full job to Surrey HPC.

Goal: 2 epochs, ~200 images, no pretrained weights — finishes in under 2 minutes on CPU. If the loss drops at all over 2 epochs, the wiring is correct. Real training happens on the cluster with `slurm/train_age.sh`.

In [ ]:
# imports — pulled from src/ so this notebook exercises the same code path as training
import sys
from pathlib import Path

# add repo root to sys.path so `from src...` works when running from notebooks/
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW

from src.data.utkface_dataset import UTKFaceDataset
from src.data.transforms import build_train_transforms, build_eval_transforms
from src.models.age_estimator import AgeEstimator
from src.models.ordinal_loss import OrdinalRegressionLoss, ordinal_logits_to_age

In [ ]:
# point this at the local UTKFace folder. update if the dataset lives somewhere else.
DATA_DIR = REPO_ROOT / 'data' / 'UTKFace'

# tiny subset — first 200 train images, first 50 val images. enough to spot a working pipeline.
train_full = UTKFaceDataset(root=str(DATA_DIR), split='train', transform=build_train_transforms())
val_full = UTKFaceDataset(root=str(DATA_DIR), split='val', transform=build_eval_transforms())

train_ds = Subset(train_full, range(min(200, len(train_full))))
val_ds = Subset(val_full, range(min(50, len(val_full))))

print(f'train subset: {len(train_ds)} images')
print(f'val subset: {len(val_ds)} images')

In [ ]:
# small batch size + no pretrained -> fast CPU smoke run
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AgeEstimator(pretrained=False).to(device)
loss_fn = OrdinalRegressionLoss().to(device)
optimizer = AdamW(model.parameters(), lr=1e-3)

print(f'device: {device}')

In [ ]:
# 2-epoch smoke run. logs train + val loss and val MAE per epoch.
for epoch in range(1, 3):
    # training pass
    model.train()
    train_loss_sum = 0.0
    train_seen = 0
    for images, ages in train_loader:
        images = images.to(device)
        ages = ages.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, ages)
        loss.backward()
        optimizer.step()
        bs = images.size(0)
        train_loss_sum += loss.item() * bs
        train_seen += bs
    train_loss = train_loss_sum / max(train_seen, 1)

    # eval pass
    model.eval()
    val_loss_sum = 0.0
    val_mae_sum = 0.0
    val_seen = 0
    with torch.no_grad():
        for images, ages in val_loader:
            images = images.to(device)
            ages = ages.to(device)
            logits = model(images)
            loss = loss_fn(logits, ages)
            pred_ages = ordinal_logits_to_age(logits)
            bs = images.size(0)
            val_loss_sum += loss.item() * bs
            val_mae_sum += (pred_ages - ages.float()).abs().sum().item()
            val_seen += bs
    val_loss = val_loss_sum / max(val_seen, 1)
    val_mae = val_mae_sum / max(val_seen, 1)

    print(f'epoch {epoch}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_mae={val_mae:.2f} years')

## What this proves

If `train_loss` drops between epoch 1 and epoch 2, gradients are flowing and the ordinal head is learning. If `val_mae` is anywhere from 15 to 30 years on this tiny subset, the decoder is producing sensible numbers — the model has not converged yet on so few samples and so few epochs, but the pipeline is correct.

Once this works locally, full training goes to Surrey HPC via `slurm/train_age.sh`.